## Script to get Quality metrics from sorted clusters

**Imports**

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from pprint import pprint
import spikeinterface.full as si
import spikeinterface.extractors as se
import spikeinterface.metrics.quality as sqm
import pandas as pd
import scipy.io as sio
import pandas as pd

**Functions**

In [2]:
None

**Specify names to access data**

In [3]:
bird = 'green_black_r3'

In [5]:
ap_folder = Path(rf'M:\StimDM\green_black\green_black_r3_g0\green_black_r3_g0_imec0\catgt_green_black_r3_g0')
ks_folder = Path(rf'M:\StimDM\green_black\green_black_r3_g0\green_black_r3_g0_imec0\catgt_green_black_r3_g0\kilosort4'); 
analyzer_path = None
phy_done = False

**Access data**

In [6]:
global_job_kwargs = dict(n_jobs=-1, chunk_duration="1s", progress_bar=True)

In [7]:
# Load recording object
ap_rec = si.read_spikeglx(ap_folder, stream_name='imec0.ap');

# Create or if existing, load sorting object
sorting_path = Path(ks_folder / 'waveform' / 'sorting')

if sorting_path.is_dir():
    sorting = si.load(ks_folder / 'waveform' / 'sorting')
else:
    # If curation in phy is done use read_phy, otherwise read_kilosort
    if phy_done:
        sorting = si.read_phy(folder_path=ks_folder)
    else:
        sorting = si.read_kilosort(folder_path=ks_folder);

In [8]:
if analyzer_path is None:
    analyzer = si.create_sorting_analyzer(sorting, ap_rec, format="binary_folder", folder= ks_folder / 'waveform', overwrite=True, **global_job_kwargs);
else:
    analyzer = si.load_sorting_analyzer(analyzer_path)

estimate_sparsity (workers: 20 processes spawn):   0%|          | 0/4521 [00:00<?, ?it/s]

**Compute needed variables**

In [9]:
#Check already calculated extensions
ext_names = analyzer.get_loaded_extension_names()
print(ext_names)

[]


In [10]:
if len(ext_names) == 0:
    analyzer.compute("random_spikes", max_spikes_per_unit=1000, save=True);
    #waveforms dimensions (num_spikes, num_samples, num_channels)
    analyzer.compute("waveforms", ms_before=1.0,ms_after=2.0, **global_job_kwargs, save=True);
    analyzer.compute("templates", operators=["average", "median", "std"], **global_job_kwargs, save=True); 
    analyzer.compute(input="spike_amplitudes", save=True)
    analyzer.compute("noise_levels", **global_job_kwargs, save=True); 

compute_waveforms (workers: 20 processes spawn):   0%|          | 0/4521 [00:00<?, ?it/s]

spike_amplitudes (no parallelization):   0%|          | 0/4521 [00:00<?, ?it/s]

noise_level (workers: 20 processes spawn):   0%|          | 0/20 [00:00<?, ?it/s]

**Compute metrics**

In [11]:
all_metrics = analyzer.compute("quality_metrics",
    metric_names=["presence_ratio", "isi_violation", "amplitude_cutoff", "snr"])

# Get metrics data as a df
metrics = all_metrics.get_data()

# Reset index and add cluster id as column - this prevents conflicts in index management later on
metrics = metrics.reset_index(names='cluster_id')

C:\Users\cgomezg\AppData\Local\anaconda3\envs\hvc\Lib\site-packages\spikeinterface\metrics\quality\misc_metrics.py:1026: UserWarning: Amplitude cutoff set to NaN for units [26, 37, 94, 132, 203, 259, 273, 290, 299, 313, 320, 321, 322, 333, 334, 338, 339, 373, 384, 385, 386, 388, 397, 399]: too few spikes (< 500).
  warnings.warn(


**Filter units by metrics**

In [12]:
isi_th = 0.3; # 0.05 strict, 0.5 loose
ampcutoff_th = 0.5;
th_mask = (metrics['isi_violations_ratio'] <= isi_th) & (metrics['amplitude_cutoff'] <= ampcutoff_th) 
noise_mask = sorting.get_property('quality') == 'noise'
conditions = [noise_mask, th_mask]
choices = ['noise', 'good']
# Check if index (cluster_id) in sorting is same as cluster_id in metrics
if (sorting.get_property('original_cluster_id') == metrics['cluster_id']).all():
    # Then add new group labels according to quality metrics thresholds
    metrics["q_group"] = np.select(conditions, choices, default='mua')
else:
    print('Mismatch in cluster_id sorting not same as metrics')

# Remove NaN values from metrics columns
metrics = metrics.fillna({'isi_violations_ratio': 1, 'amplitude_cutoff': 1})

# Round metrics columns to max 6 decimals
metrics = metrics.round({'isi_violations_ratio': 6, 'amplitude_cutoff': 6})

**Update sorting properties adding new label groups for each cluster id**

In [13]:
sorting.set_property('q_group', metrics["q_group"])

**Save updated sorting**

In [14]:
# Make sure saving is done in correct specific folder for sorting and analyzer
sorting.save(folder=ks_folder / 'waveform' / 'sorting', overwrite=True)

NumpyFolder (NumpyFolderSorting): 408 units - 1 segments - 30.0kHz

**Save quality metrics data to.tsv files for phy**

In [18]:
metrics[['cluster_id', 'q_group']].to_csv(ks_folder / 'cluster_q_group.tsv', sep='\t', index=False)
metrics[['cluster_id', 'isi_violations_ratio']].to_csv(ks_folder / 'cluster_isi.tsv', sep='\t', index=False)
metrics[['cluster_id', 'amplitude_cutoff']].to_csv(ks_folder / 'cluster_aco.tsv', sep='\t', index=False)

**FIN**